#### Import Libraries and General Settings

In [3]:
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
)
import optuna

from optuna.visualization.matplotlib import plot_optimization_history
optuna.logging.set_verbosity(optuna.logging.WARNING)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    precision_recall_curve,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

import xgboost, lightgbm, sklearn

print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("Optuna:", optuna.__version__)
print("XGBoost:", xgboost.__version__)
print("LightGBM:", lightgbm.__version__)
print("Sklearn:", sklearn.__version__)

Python: 3.11.14 (main, Oct 10 2025, 08:54:03) [GCC 11.4.0]
numpy: 2.4.1
pandas: 3.0.0
Optuna: 4.7.0
XGBoost: 3.1.3
LightGBM: 4.6.0
Sklearn: 1.4.2


#### Load dataset

In [4]:
df = pd.read_csv('dataset/creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


#### Statistical summary of the columns

In [5]:
df.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.175161e-15,3.384974e-16,-1.379537e-15,2.094852e-15,1.021879e-15,1.494498e-15,-5.620335e-16,1.149614e-16,-2.414189e-15,...,1.628620e-16,-3.576577e-16,2.618565e-16,4.473914e-15,5.109395e-16,1.686100e-15,-3.661401e-16,-1.227452e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


#### Number of rows, columns, data types, number of non-null values

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-nu

#### Target information

In [7]:
fraud = df.Class.value_counts()
print(f"Total no fraud: {fraud[0]}")
print(f"Total fraud: {fraud[1]}")
percentaje = (fraud[1] * 100) / fraud[0]
print(f"Percentaje fraud: {percentaje:.2f}%")

Total no fraud: 284315
Total fraud: 492
Percentaje fraud: 0.17%


#### Separate features / target

In [8]:
X = df.drop('Class', axis=1)
y = df.Class

#### Split train/test

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

#### Evaluate the threshold within each CV

In [10]:
N_FOLDS = 5

def evaluate_fraud_model_with_threshold_cv(
    model,
    X,
    y,
    cv_splits=N_FOLDS,
    optimize_for="recall_at_precision",
    precision_target=0.90,
    recall_target=0.70,
):
    """
    Evaluación robusta para fraude extremo (0.17%)
    con optimización de threshold dentro de cada fold.
    """

    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    metrics = []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        if isinstance(model, LGBMClassifier):
            model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
        elif isinstance(model, XGBClassifier):
            model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        else:
            model.fit(X_train, y_train)

        # Probabilidades
        train_proba = model.predict_proba(X_train)[:, 1]
        test_proba = model.predict_proba(X_test)[:, 1]

        # ===== OPTIMIZACIÓN DE THRESHOLD (solo con TRAIN) =====
        precisions, recalls, thresholds = precision_recall_curve(y_train, train_proba)

        best_threshold = 0.5

        if optimize_for == "f1":
            f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
            best_idx = np.argmax(f1_scores)
            if best_idx == 0:
                best_threshold = 0.0
            else:
                best_threshold = thresholds[best_idx - 1]

        elif optimize_for == "recall_at_precision":
            valid_idxs = np.where(precisions >= precision_target)[0]
            if len(valid_idxs) > 0:
                best_idx = valid_idxs[np.argmax(recalls[valid_idxs])]
                best_threshold = thresholds[max(best_idx - 1, 0)]
            else:
                # fallback: usar threshold que maximice F1
                f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
                best_idx = np.argmax(f1_scores)
                best_threshold = thresholds[max(best_idx - 1, 0)]

        elif optimize_for == "precision_at_recall":
            valid_idxs = np.where(recalls >= recall_target)[0]
            if len(valid_idxs) > 0:
                best_idx = valid_idxs[np.argmax(precisions[valid_idxs])]
                best_threshold = thresholds[max(best_idx - 1, 0)]
            else:
                # fallback: usar threshold que maximice F1
                f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
                best_idx = np.argmax(f1_scores)
                best_threshold = thresholds[max(best_idx - 1, 0)]

        # ===== Evaluación en TEST con threshold optimizado =====
        y_pred_test = (test_proba >= best_threshold).astype(int)

        fold_metrics = {
            "ROC AUC": roc_auc_score(y_test, test_proba),
            "PR AUC": average_precision_score(y_test, test_proba),
            "Precision": precision_score(y_test, y_pred_test, zero_division=0),
            "Recall": recall_score(y_test, y_pred_test, zero_division=0),
            "F1": f1_score(y_test, y_pred_test, zero_division=0),
            "Threshold": best_threshold,
            "Positives in fold": int(y_test.sum()),
        }

        metrics.append(fold_metrics)

    # ===== Agregación final =====
    results = {
        metric: np.mean([fold[metric] for fold in metrics])
        for metric in metrics[0].keys()
    }

    results["Threshold std"] = np.std([fold["Threshold"] for fold in metrics])

    return results

#### Define ranges for each parameter of each model
*Optuna was used to optimize each model*

*Selected models:*
- **XGBRegressor**
- **LGBMRegressor**

In [11]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# -------------------------
# Objective XGBClassifier
# -------------------------


def objective_xgb(trial):

    params = {
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 3),
        "n_estimators": trial.suggest_int("n_estimators", 1100, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.12, 0.14),
        "max_depth": trial.suggest_int("max_depth", 8, 12),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.2, 1.6),
        "subsample": trial.suggest_float("subsample", 0.55, 0.70),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.5),
        "gamma": trial.suggest_float("gamma", 0.8, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.9, 1.3),
        "reg_lambda": trial.suggest_float("reg_lambda", 5.5, 7.5),
        "objective": "binary:logistic",
        "scale_pos_weight": pos_weight,
        "early_stopping_rounds": 50,
        "eval_metric": "aucpr",
        "random_state": 42,
        "n_jobs": -1,
    }
    model = XGBClassifier(**params)
    results = evaluate_fraud_model_with_threshold_cv(
        model, X, y, optimize_for="recall_at_precision", precision_target=0.90
    )
    return results["Recall"]


# -------------------------
# Objective LGBMClassifier
# -------------------------


def objective_lgb(trial):

    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.014, 0.016, log=True),
        "scale_pos_weight": pos_weight,
        "num_leaves": 121,
        "min_child_samples": 285,
        "subsample": 0.854,
        "colsample_bytree": 0.652,
        "min_split_gain": 0.022,
        "reg_lambda": 3.09,
        "reg_alpha": 0.55,
        "objective": "binary",
        "metric": "average_precision",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "n_estimators": 5000,
        "max_depth": 7,
        "random_state": 42,
        "n_jobs": -1,
    }

    model = LGBMClassifier(**params)
    results = evaluate_fraud_model_with_threshold_cv(
        model, X, y, optimize_for="recall_at_precision", precision_target=0.90
    )
    return results["Recall"]

#### Create studies and optimize

In [12]:
studies = {}
objectives = {
    "Xgb": objective_xgb,
    "Lgb": objective_lgb,
}
n_trials = 8

for name, obj in objectives.items():
    print(f"\nOptimizing {name} ...")
    study = optuna.create_study(
        direction="maximize",
        study_name=f"{name}",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(obj, n_trials=n_trials, show_progress_bar=True)
    studies[name] = study
    print(f"Best params: {name}")
    for k, v in study.best_trial.params.items():
        print(f"  {k}: {v}")


Optimizing Xgb ...


Best trial: 7. Best value: 0.853618: 100%|██████████| 8/8 [06:30<00:00, 48.86s/it]


Best params: Xgb
  max_delta_step: 3
  n_estimators: 1179
  learning_rate: 0.12011044234247205
  max_depth: 12
  min_child_weight: 1.4827429375390468
  subsample: 0.6593510752061481
  colsample_bytree: 0.4771270346685946
  gamma: 0.807404465173409
  reg_alpha: 1.043386291417709
  reg_lambda: 5.731738119050259

Optimizing Lgb ...


Best trial: 1. Best value: 0.85974: 100%|██████████| 8/8 [1:22:15<00:00, 616.96s/it]

Best params: Lgb
  learning_rate: 0.015895046740515752


#### XGBClassifier Optimization History

In [ ]:
plot_optimization_history(studies['Xgb'])

#### LGBMClassifier Optimization History

In [ ]:
plot_optimization_history(studies['Lgb'])

#### Extract final metrics from each best trial

In [109]:
results_list = []

for name, study in studies.items():
    best_trial = study.best_trial
    if name == "Xgb":
        model = XGBClassifier(
            **best_trial.params,
            objective="binary:logistic",
            scale_pos_weight=pos_weight,
            early_stopping_rounds=50,
            eval_metric="aucpr",
            random_state=42,
            n_jobs=-1
        )

    else:
        model = LGBMClassifier(
            **best_trial.params,
            scale_pos_weight=pos_weight,
            num_leaves=121,
            min_child_samples=285,
            subsample=0.854,
            colsample_bytree=0.652,
            min_split_gain=0.022,
            reg_lambda=3.09,
            reg_alpha=0.55,
            objective="binary",
            metric="average_precision",
            boosting_type="gbdt",
            verbosity=-1,
            n_estimators=5000,
            max_depth=7,
            random_state=42,
            n_jobs=-1
        )

    res = evaluate_fraud_model_with_threshold_cv(model, X, y)
    res["Model"] = name
    results_list.append(res)

#### Display metrics

In [ ]:
results_df = pd.DataFrame(results_list)
results_df = results_df[
    [
        "Model",
        "ROC AUC",
        "PR AUC",
        "Precision",
        "Recall",
        "F1",
        "Threshold",
        "Positives in fold",
    ]
]

cols_highlight_max = ["ROC AUC", "PR AUC", "Precision", "Recall", "F1", "Threshold"]

styled_df = (
    results_df.style.highlight_max(subset=cols_highlight_max, color="#00441b").format(
        {
            "ROC AUC": "{:.4f}",
            "PR AUC": "{:.4f}",
            "Precision": "{:.4f}",
            "Recall": "{:.4f}",
            "F1": "{:.4f}",
            "Threshold": "{:.4f}",
            "Positives in fold": "{:.1f}",
        }
    )
)
styled_df

,Model,ROC AUC,PR AUC,Precision,Recall,F1,Threshold,Positives in fold
0,Xgb,0.9810,0.8627,0.8140,0.8536,0.8330,0.1769,98.4
1,Lgb,0.9757,0.8597,0.7263,0.8597,0.7872,0.0537,98.4


#### Technical Analysis

1️⃣ PR AUC (the key metric with a 0.17% positive rate)

Xgb: 0.8627<br>
Lgb: 0.8597

They're almost the same, but Xgb wins..<br >
With 0.17% fraud, PR AUC > 0.85 is VERY strong.

2️⃣ Precision

Xgb: 81.4%<br >
Lgb: 72.6%

This is important.<br >
In fraud, each false positive costs an operation.<br >
Xgb generates MANY fewer false positives.<br >
Large difference → +9 percentage points.

3️⃣ Recall

Xgb: 85.36%<br >
Lgb: 85.97%

Virtually the same.<br >
Lgb recovers only 0.6% more fraud.<br > 
That's marginal compared to the sharp drop in precision.

4️⃣ Threshold

Xgb: 0.1769<br >
Lgb: 0.0537

Lgb needs a much lower threshold to achieve similar recall → that explains its worse accuracy.<br >
This usually means:<br >
- Xgb separates probabilities better<br >
- Better implicit calibration<br >
- Better risk ranking

🎯 Conclusión

👉 Xgb is clearly superior in this scenario.<br >
Because:
- It maintains high recall<br >
- Much better accuracy<br >
- Better F1<br >
- Better PR AUC<br >
- Better probabilistic separation (higher threshold)

In production, this translates to:<br >
- Fewer unnecessary tickets<br >
- Less customer friction<br >
- Lower operating costs<br >
- Same level of protection

#### Instantiate the best model

In [111]:
xgb = studies["Xgb"]
model_xgb = XGBClassifier(
    **xgb.best_params,
    objective="binary:logistic",
    scale_pos_weight=pos_weight,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)
model_xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.4771270346685946, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='aucpr', feature_types=None, feature_weights=None,
              gamma=0.807404465173409, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.12011044234247205,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=3, max_depth=12, max_leaves=None,
              min_child_weight=1.4827429375390468, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1179,
              n_jobs=-1, num_parallel_tree=None, ...)

#### Show predictions

In [112]:
y_pred = model_xgb.predict(X_test)
predictions = pd.DataFrame()
predictions['Actual'] = y_test
predictions['Predicted'] = y_pred
predictions.sample(10)

,Actual,Predicted
147167,0,0
150921,0,0
139941,0,0
157558,0,0
183571,0,0
147983,0,0
53647,0,0
190533,0,0
60202,0,0
251668,0,0


0 = No fraud<br>
1 = Fraud